# Assignment 05: Logistic Regression (100 points)

**Unit**: ML1 Supervised Learning (AI 300)  
**Topics**: Sigmoid function, binary cross-entropy, gradient descent, decision boundary, L2 regularization

---

## Background

**Logistic regression** models $P(y=1 \mid x) = \sigma(w^Tx)$ where $\sigma(z) = \frac{1}{1 + e^{-z}}$.

The **binary cross-entropy** (BCE) loss is:
$$\mathcal{L}(w) = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log \hat{y}_i + (1 - y_i)\log(1 - \hat{y}_i)\right]$$

The gradient is: $\nabla_w \mathcal{L} = \frac{1}{n}X^T(\hat{y} - y)$ where $\hat{y} = \sigma(Xw)$.

### Notation

| Symbol | Shape | Description |
|--------|-------|-------------|
| $X$ | $(n, d)$ | Design matrix (includes bias column) |
| $w$ | $(d,)$ | Weight vector |
| $\hat{y}$ | $(n,)$ | Predicted probabilities $\sigma(Xw)$ |
| $\sigma$ | | Sigmoid function |

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

> **WARNING !!!**
>
> - Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
>     - **As a part of your final solution.**
>     - **Temporarily import something to assist you to get a solution.**
>
>     **Rule of thumb:** Each part has its particular purpose to intentionally test you something. Do not attempt to find a shortcut to circumvent the rule.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
Dataset: 2D binary classification.
"""
n = 200
X_pos = np.random.randn(n // 2, 2) + np.array([2, 2])      # (100, 2)
X_neg = np.random.randn(n // 2, 2) + np.array([-1, -1])    # (100, 2)
X_raw = np.vstack([X_pos, X_neg])                            # (200, 2)
X = np.hstack([np.ones((n, 1)), X_raw])                     # (200, 3)  bias trick
y = np.array([1] * (n // 2) + [0] * (n // 2), dtype=float) # (200,)
d = X.shape[1]                                               # 3

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class balance: {y.mean():.2f}")

---

## Part 1 (15 points, coding task)

**Implement the numerically stable sigmoid function.**

Naive $\sigma(z) = \frac{1}{1+e^{-z}}$ overflows for large negative $z$. Use:
- For $z \geq 0$: $\sigma(z) = \frac{1}{1 + e^{-z}}$
- For $z < 0$: $\sigma(z) = \frac{e^z}{1 + e^z}$

Must be vectorized (no loops).

*Reasoning is not required.*

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """
    Numerically stable sigmoid function.

    Args:
        z: array of any shape

    Returns:
        Array of same shape, values in (0, 1)
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
assert np.allclose(sigmoid(np.array([0.0])), 0.5), "sigma(0) should be 0.5"
z_test = np.linspace(-10, 10, 100)
s_test = sigmoid(z_test)
assert np.all(s_test > 0) and np.all(s_test < 1), "sigma must be in (0, 1)"
assert np.allclose(s_test + sigmoid(-z_test), 1.0), "sigma(z) + sigma(-z) = 1"

# Numerical stability with extreme values
assert sigmoid(np.array([1000.0]))[0] == 1.0, "sigma(1000) should be 1.0"
assert sigmoid(np.array([-1000.0]))[0] == 0.0, "sigma(-1000) should be 0.0"
print(f"sigma(1000) = {sigmoid(np.array([1000.0]))[0]}")
print(f"sigma(-1000) = {sigmoid(np.array([-1000.0]))[0]}")
print("Part 1 passed.")

""" END OF THIS PART """

---

## Part 2 (15 points, coding task)

**Implement binary cross-entropy loss and its gradient.**

- BCE: $\mathcal{L} = -\frac{1}{n}\sum\left[y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\right]$
- Gradient: $\nabla_w \mathcal{L} = \frac{1}{n}X^T(\hat{y} - y)$

Clip $\hat{y}$ to $[\epsilon, 1-\epsilon]$ to avoid $\log(0)$.

*Reasoning is not required.*

In [ ]:
def bce_loss(y: np.ndarray, y_hat: np.ndarray, eps: float = 1e-12) -> float:
    """
    Binary cross-entropy loss.

    Args:
        y:     (n,) true binary labels {0, 1}
        y_hat: (n,) predicted probabilities
        eps:   clipping epsilon

    Returns:
        Scalar loss value
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass


def bce_gradient(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    """
    Gradient of BCE loss w.r.t. w: (1/n) X^T (sigmoid(Xw) - y)

    Args:
        X: (n, d), y: (n,), w: (d,)

    Returns:
        grad: (d,) gradient vector
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
# Loss sanity checks
y_perfect = np.array([0.99, 0.01, 0.99, 0.01])
y_bad = np.array([0.01, 0.99, 0.01, 0.99])
y_lbl = np.array([1, 0, 1, 0], dtype=float)
assert bce_loss(y_lbl, y_perfect) < 0.1, "Good predictions should give low loss"
assert bce_loss(y_lbl, y_bad) > 2.0, "Bad predictions should give high loss"

# Gradient check (numerical)
w_test = np.random.randn(d) * 0.1
analytical = bce_gradient(X, y, w_test)
assert analytical.shape == (d,)
eps = 1e-5
num_grad = np.zeros(d)
for j in range(d):
    wp = w_test.copy(); wp[j] += eps
    wm = w_test.copy(); wm[j] -= eps
    num_grad[j] = (bce_loss(y, sigmoid(X @ wp)) - bce_loss(y, sigmoid(X @ wm))) / (2 * eps)
assert np.allclose(analytical, num_grad, atol=1e-5), "Gradient mismatch"
print(f"Max gradient error: {np.max(np.abs(analytical - num_grad)):.2e}")
print("Part 2 passed.")

""" END OF THIS PART """

---

## Part 3 (20 points, coding task)

**Implement logistic regression via gradient descent.**

Update rule: $w^{(t+1)} = w^{(t)} - \eta \cdot \nabla_w \mathcal{L}(w^{(t)})$

Initialize $w = \mathbf{0}$. Record the loss at every step.

*Reasoning is not required.*

In [ ]:
def logistic_regression(
    X: np.ndarray,       # (n, d)
    y: np.ndarray,       # (n,) binary labels
    lr: float = 0.1,
    n_steps: int = 1000,
) -> tuple:
    """
    Train logistic regression via gradient descent.

    Args:
        X: (n, d) design matrix (with bias column)
        y: (n,) binary labels
        lr: learning rate
        n_steps: number of gradient steps

    Returns:
        w: (d,) final weight vector
        losses: list of length n_steps, BCE loss at each step
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
w_lr, losses_lr = logistic_regression(X, y, lr=0.1, n_steps=2000)
print(f"Weights: {w_lr.round(4)}")
assert w_lr.shape == (d,)
assert len(losses_lr) == 2000

# Check convergence: loss should decrease
assert losses_lr[-1] < losses_lr[0], "Loss should decrease"

# Check accuracy
y_pred_lr = (sigmoid(X @ w_lr) >= 0.5).astype(int)
acc_lr = np.mean(y_pred_lr == y)
print(f"Accuracy: {acc_lr:.4f}")
assert acc_lr > 0.90, f"Accuracy too low: {acc_lr:.4f}"
print("Part 3 passed.")

""" END OF THIS PART """

---

## Part 4 (20 points, coding task)

**Visualize the training process and decision boundary.**

Create a 2x2 subplot grid (figure size 14x10):

1. **Top-left**: Loss curve (BCE loss vs iteration).
2. **Top-right**: Data points colored by class with the decision boundary line $w_0 + w_1 x_1 + w_2 x_2 = 0$.
3. **Bottom-left**: Filled contour plot of $P(y=1 \mid x)$ across the 2D space, with data points and decision boundary overlaid.
4. **Bottom-right**: Histogram of predicted probabilities for each class (two overlapping histograms).

*Reasoning is not required.*

In [ ]:
### WRITE YOUR SOLUTION HERE ###

pass

""" END OF THIS PART """

---

### Regularization in Logistic Regression

Adding L2 regularization penalizes large weights: $\mathcal{L}_{\text{reg}} = \mathcal{L}_{\text{BCE}} + \lambda \|w\|_2^2$. The gradient becomes $\nabla_w \mathcal{L}_{\text{reg}} = \frac{1}{n}X^T(\hat{y} - y) + 2\lambda w$. Convention: do not regularize the bias term.

---

## Part 5 (15 points, coding task)

**Implement L2-regularized logistic regression.**

1. Add the regularization term to both loss and gradient. Do **not** regularize the bias ($w_0$).
2. Train with $\lambda \in \{0, 0.1, 1.0, 10.0\}$.
3. Create a 1x4 subplot showing the decision boundary for each $\lambda$. Title each with $\lambda$ and accuracy.

*Reasoning is not required.*

In [ ]:
def logistic_regression_l2(
    X: np.ndarray,       # (n, d)
    y: np.ndarray,       # (n,)
    lr: float = 0.1,
    lam: float = 0.1,
    n_steps: int = 1000,
) -> tuple:
    """
    L2-regularized logistic regression.

    Loss: BCE + lambda * ||w[1:]||^2  (don't regularize bias w[0])
    Gradient: (1/n) X^T (y_hat - y) + 2*lambda * [0, w[1], ..., w[d-1]]

    Returns:
        w: (d,) final weights
        losses: list of losses
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass


### WRITE YOUR SOLUTION HERE ###
# Train with lambda in {0, 0.1, 1.0, 10.0}
# Plot decision boundaries for each

pass

""" END OF THIS PART """

---

## Part 6 (15 points, non-coding task)

**Derivation and analysis questions.**

1. Derive the gradient $\nabla_w \mathcal{L}_{\text{BCE}} = \frac{1}{n}X^T(\hat{y} - y)$. Start from the per-sample loss, apply the chain rule using $\frac{d\sigma}{dz} = \sigma(z)(1-\sigma(z))$, and simplify.

2. Prove that the BCE loss is **convex** by showing the Hessian $H = \frac{1}{n}X^T S X$ is positive semi-definite, where $S = \text{diag}(\hat{y}_i(1-\hat{y}_i))$.

3. Explain why we use cross-entropy instead of MSE for classification. What goes wrong with $\mathcal{L}_{\text{MSE}} = \frac{1}{n}\|\sigma(Xw) - y\|^2$ in terms of gradient magnitude when $\hat{y}$ is far from $y$?

*Reasoning is required.*

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """